# client model 정의

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiModalClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()

        # 이미지 인코더
        self.img_fc1 = nn.Linear(input_dim, hidden_dim)

        # 텍스트 인코더
        self.txt_fc1 = nn.Linear(input_dim, hidden_dim)

        # Gating Mechanism (Forget Gate & Input Gate)
        self.forget_gate = nn.Linear(hidden_dim * 2, hidden_dim)  # Forget Gate 적용
        self.input_gate = nn.Linear(hidden_dim * 2, hidden_dim)  # Input Gate 적용

        # Fully-Connected Layer (최종 Classification)
        self.fusion_fc = nn.Linear(hidden_dim, num_classes)

        # 초기 Forget Gate의 상태 (Z'_{t-1} = 0으로 설정)
        self.prev_Z = torch.zeros(1, hidden_dim)

    def forward(self, img_input=None, txt_input=None):
        """
        이미지 & 텍스트 입력을 받아서 각각 Representation을 만든 후, Forget Gate & Input Gate를 활용한 Late Fusion 수행.
        """
        img_representation, txt_representation = None, None

        # 이미지 Representation 처리
        if img_input is not None:
            img_x = F.relu(self.img_fc1(img_input))
            img_representation = img_x

        # 텍스트 Representation 처리
        if txt_input is not None:
            txt_x = F.relu(self.txt_fc1(txt_input))
            txt_representation = txt_x

        # Late Fusion 적용 (Gating Mechanism 사용)
        if img_representation is not None and txt_representation is not None:
            fusion_representation = torch.cat((img_representation, txt_representation), dim=1)

            # Forget Gate를 통해 이전 시점 정보 반영
            forget_weight = torch.sigmoid(self.forget_gate(fusion_representation))
            updated_rep = forget_weight * self.prev_Z + (1 - forget_weight) * fusion_representation

            # Input Gate 적용하여 현재 정보 반영
            input_weight = torch.sigmoid(self.input_gate(fusion_representation))
            updated_rep = input_weight * updated_rep + (1 - input_weight) * fusion_representation

            # 현재 상태를 다음 시점의 Z'_{t-1}로 업데이트
            self.prev_Z = updated_rep.detach()

        elif img_representation is not None:
            updated_rep = img_representation
        elif txt_representation is not None:
            updated_rep = txt_representation
        else:
            raise ValueError("No input provided!")

        # 최종 Classification 수행
        logits = self.fusion_fc(updated_rep)

        return logits

: 

# Teacher Model을 선택하는 기준
**1. Global Test Accuracy 기준 (가장 간단한 방법)**

각 클라이언트 모델의 Validation Accuracy를 평가하여 가장 높은 상위 K개 모델을 Teacher로 선정

**2. Representation Quality 기준 (더 정밀한 방법)**

Lev1, Lev2 모델들이 Feature Representation을 얼마나 잘 생성하는지 측정
Cosine Similarity나 Euclidean Distance를 활용하여, Representation의 질이 높은 모델을 Teacher로 선정


In [ ]:
client_models = [...]  # 전체 클라이언트 모델 리스트
client_accuracies = [evaluate_model(model) for model in client_models]  # Accuracy 측정

K = 5  # Teacher로 선정할 모델 개수
sorted_indices = sorted(range(len(client_accuracies)), key=lambda i: client_accuracies[i], reverse=True)
top_k_indices = sorted_indices[:K]
teacher_models = [client_models[i] for i in top_k_indices]  # Top K Teacher 모델 선정


# Lev3 모델 업데이트 대상 선정 기준
###가장 Representation Quality가 낮은 모델부터 업데이트

Teacher Model과 **Representation Distance가 가장 먼 Lev3 모델**부터 학습

예: Cosine Similarity를 측정하여, Teacher와 가장 다른 모델들을 우선적으로 업데이트
클러스터링을 활용하여 일부 모델만 업데이트

Lev3 모델들을 Feature Space에서 클러스터링하고,각 클러스터에서 대표적인 모델만 업데이트

너무 비슷한 모델끼리는 업데이트하지 않음 → 비효율적인 중복 학습 방지

In [ ]:
lev3_representations = [get_representation(model) for model in lev3_models]  # 각 모델의 벡터 추출
teacher_representations = [get_representation(model) for model in teacher_models]
mean_teacher_representation = torch.mean(torch.stack(teacher_representations), dim=0)

In [ ]:
from scipy.spatial.distance import cosine

distances = [cosine(rep, mean_teacher_representation) for rep in lev3_representations]
sorted_indices = sorted(range(len(distances)), key=lambda i: distances[i], reverse=True)  # 거리가 큰 순 정렬

K_update = 5  # 업데이트할 Lev3 모델 개수
update_indices = sorted_indices[:K_update]
lev3_models_to_update = [lev3_models[i] for i in update_indices]  # 업데이트 대상 선정

# Lev1, Lev2 모델을 Teacher, Lev3 모델을 Student로 설정

1. Lev1, Lev2는 Teacher Model (고성능 모델)

2. Lev3는 Student Model (KD로 학습할 모델)

Teacher Model에서 나온 Representation을 Student Model이 학습하도록 설계

# Knowledge Distillation Loss 정의

Teacher Model의 Representation과 Student Model의 Representation을 최대한 정렬

Student Model이 Teacher의 분류 결과를 따르도록 학습

KL Divergence를 활용하여 두 모델 간의 Soft Target 차이를 최소화


In [ ]:
def knowledge_distillation_loss(student_logits, teacher_logits, temperature=2.0):
    """
    Teacher Model과 Student Model 간 Knowledge Distillation Loss 계산
    """
    student_probs = F.log_softmax(student_logits / temperature, dim=-1)
    teacher_probs = F.softmax(teacher_logits / temperature, dim=-1)  # Teacher의 Soft Target

    loss = F.kl_div(student_probs, teacher_probs, reduction="batchmean")  # KL Divergence Loss
    return loss


In [ ]:
teacher1_rep, teacher1_logits = lev1_model(input_data)  # Lev1 (Teacher) 그룹 중 가장 성능이 좋은 모델 선정(loss 기준)
teacher2_rep, teacher2_logits = lev2_model(input_data)  # Lev2 (Teacher)
student_rep, student_logits = lev3_model(input_data)  # Lev3 (Student)

rep_loss = F.mse_loss(student_rep, (teacher1_rep + teacher2_rep) / 2)  # 평균 Representation 사용
logits_loss = knowledge_distillation_loss(student_logits, (teacher1_logits + teacher2_logits) / 2)
total_loss = rep_loss + logits_loss  # Representation + Logits Loss
optimizer = torch.optim.Adam(lev3_model.parameters(), lr=0.001)
optimizer.zero_grad()
total_loss.backward()
optimizer.step()

updated_representation, _ = lev3_model(input_data)  # 새롭게 학습된 Lev3 모델에서 벡터 추출

# K-Means 적용 방식

1. 클라이언트 그룹 정의

- img-text: 이미지 & 텍스트 둘 다 있는 그룹
- img-only: 이미지 벡터만 있는 그룹
- text-only: 텍스트 벡터만 있는 그룹

2. 각 그룹 내에서 모델 레벨(Lev1, Lev2, Lev3) 별로 벡터를 추출

- Lev1 모델에서 이미지 벡터 / 텍스트 벡터 추출
- Lev2 모델에서 이미지 벡터 / 텍스트 벡터 추출
- Lev3 모델에서 이미지 벡터 / 텍스트 벡터 추출

3. 각 그룹 내에서 이미지 벡터 / 텍스트 벡터 별로 K-Means 수행

- img-text 그룹: 이미지 벡터와 텍스트 벡터 각각 K-Means 적용
- img-only 그룹: 이미지 벡터만 K-Means 적용
- text-only 그룹: 텍스트 벡터만 K-Means 적용

In [ ]:
import torch
import numpy as np
from sklearn.cluster import KMeans

def kmeans_clustering(vectors, num_clusters=5):
    """
    주어진 벡터들에 대해 K-Means Clustering을 수행하고 클러스터의 중심 벡터를 반환.

    Args:
        vectors (torch.Tensor): (num_samples, feature_dim) 형태의 벡터 집합
        num_clusters (int): K-Means에서 생성할 클러스터 개수

    Returns:
        cluster_labels (np.array): 각 샘플의 클러스터 할당 결과
        centroids (np.array): 각 클러스터의 중심 벡터
    """
    if len(vectors) == 0:  # 데이터가 없으면 None 반환
        return None, None

    vectors_np = vectors.cpu().detach().numpy()  # PyTorch Tensor → NumPy 변환
    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)  # KMeans 모델
    cluster_labels = kmeans.fit_predict(vectors_np)  # 클러스터링 수행
    centroids = kmeans.cluster_centers_  # 클러스터 중심 벡터

    return cluster_labels, centroids


In [ ]:
# 예제 데이터: 그룹별로 이미지 & 텍스트 벡터 저장
group_vectors = {
    "img-text": {
        "lev1": {"img": torch.rand(50, 512), "txt": torch.rand(50, 512)},  # 50개의 벡터
        "lev2": {"img": torch.rand(40, 512), "txt": torch.rand(40, 512)},
        "lev3": {"img": torch.rand(30, 512), "txt": torch.rand(30, 512)},
    },
    "img-only": {
        "lev1": {"img": torch.rand(50, 512), "txt": None},
        "lev2": {"img": torch.rand(40, 512), "txt": None},
        "lev3": {"img": torch.rand(30, 512), "txt": None},
    },
    "text-only": {
        "lev1": {"img": None, "txt": torch.rand(50, 512)},
        "lev2": {"img": None, "txt": torch.rand(40, 512)},
        "lev3": {"img": None, "txt": torch.rand(30, 512)},
    },
}

# K 값 설정 (클러스터 개수)
K = 5

# 결과 저장할 딕셔너리
clustering_results = {}

# 그룹별 K-Means Clustering 수행
for group, levels in group_vectors.items():
    clustering_results[group] = {"img": None, "txt": None}  # 결과 저장

    # 이미지 벡터 전체 모으기
    img_vectors = torch.cat([levels[lev]["img"] for lev in levels if levels[lev]["img"] is not None], dim=0) \
        if any(levels[lev]["img"] is not None for lev in levels) else None

    # 텍스트 벡터 전체 모으기
    txt_vectors = torch.cat([levels[lev]["txt"] for lev in levels if levels[lev]["txt"] is not None], dim=0) \
        if any(levels[lev]["txt"] is not None for lev in levels) else None

    # 이미지 벡터 클러스터링 수행
    if img_vectors is not None:
        img_labels, img_centroids = kmeans_clustering(img_vectors, num_clusters=K)
        clustering_results[group]["img"] = {"labels": img_labels, "centroids": img_centroids}

    # 텍스트 벡터 클러스터링 수행
    if txt_vectors is not None:
        txt_labels, txt_centroids = kmeans_clustering(txt_vectors, num_clusters=K)
        clustering_results[group]["txt"] = {"labels": txt_labels, "centroids": txt_centroids}

# 클러스터링 결과 출력
for group, results in clustering_results.items():
    print(f"{group} 그룹 K-Means 결과")
    if results["img"] is not None:
        print(f"   - 이미지 클러스터 중심 벡터 크기: {results['img']['centroids'].shape}")
    if results["txt"] is not None:
        print(f"   - 텍스트 클러스터 중심 벡터 크기: {results['txt']['centroids'].shape}")


1. K-Means 결과에서 클러스터 개수를 확인하여, 해당 개수만큼 Transformer 모델을 생성
- 각 클러스터마다 개별 Transformer를 사용하여 Cross-Attention 수행

2. 각 클러스터의 중심 벡터(Centroid)를 Query로 사용하여 Cross-Attention 적용
- Query: 이미지 클러스터의 Centroid
- Key & Value: 해당 클러스터의 텍스트 벡터들

3. Transformer 모델이 클러스터 내 벡터를 학습한 후, 최종 Output 𝑍를 생성

# 클러스터 개수 맞추기

In [ ]:
import torch
import numpy as np
from sklearn.cluster import KMeans

def match_cluster_count(img_clusters, txt_clusters):
    """
    Args:
        img_clusters (int): 이미지 클러스터 개수
        txt_clusters (int): 텍스트 클러스터 개수
    Returns:
        matched_clusters (int): 최종 조정된 클러스터 개수
    """
    matched_clusters = min(img_clusters, txt_clusters)  # 작은 쪽에 맞추기
    return matched_clusters

In [ ]:
# 클러스터링 결과에서 개수 확인
num_img_clusters = clustering_results["img-text"]["img"]["centroids"].shape[0] if clustering_results["img-text"]["img"] else 0
num_txt_clusters = clustering_results["img-text"]["txt"]["centroids"].shape[0] if clustering_results["img-text"]["txt"] else 0

# 이미지와 텍스트 클러스터 개수 맞추기
matched_cluster_count = match_cluster_count(num_img_clusters, num_txt_clusters)

# 클러스터 개수 조정 후 데이터 저장
img_centroids = clustering_results["img-text"]["img"]["centroids"][:matched_cluster_count] if clustering_results["img-text"]["img"] else None
txt_vectors = clustering_results["img-text"]["txt"]["labels"][:matched_cluster_count] if clustering_results["img-text"]["txt"] else None

print(f"클러스터 개수: {matched_cluster_count}")


# Cosine Similarity를 이용한 페어링
이미지 클러스터 중심 벡터 $C_i$와 텍스트 클러스터 중심 벡터 $C_t$ 간의 유사도를 측정

Cosine Similarity를 사용하여, 가장 유사한 페어를 우선적으로 매칭
이를 위해 헝가리 알고리즘 (Hungarian Algorithm, Munkres Algorithm) 을 적용
→ 최적의 1:1 클러스터 페어링을 보장

In [ ]:
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cosine
import numpy as np
import torch

def match_image_text_clusters(img_centroids, txt_centroids):
    """
    Args:
        img_centroids (np.array): 이미지 클러스터 중심 벡터 (num_clusters, feature_dim)
        txt_centroids (np.array): 텍스트 클러스터 중심 벡터 (num_clusters, feature_dim)
    Returns:
        matched_pairs (list of tuples): 최적의 이미지-텍스트 클러스터 페어링 리스트 [(img_idx, txt_idx), ...]
    """
    num_clusters = img_centroids.shape[0]

    # Cosine Distance Matrix 생성
    cost_matrix = np.zeros((num_clusters, num_clusters))
    for i in range(num_clusters):
        for j in range(num_clusters):
            cost_matrix[i, j] = cosine(img_centroids[i], txt_centroids[j])  # 유사도 기반 거리 측정

    # 헝가리 알고리즘 적용 (최적의 1:1 매칭)
    img_indices, txt_indices = linear_sum_assignment(cost_matrix)

    # 매칭 결과 반환
    matched_pairs = list(zip(img_indices, txt_indices))
    return matched_pairs

In [ ]:
# 클러스터 개수 조정 후 Centroid 가져오기
img_centroids = clustering_results["img-text"]["img"]["centroids"][:matched_cluster_count]
txt_centroids = clustering_results["img-text"]["txt"]["centroids"][:matched_cluster_count]

# NumPy 변환
img_centroids_np = np.array(img_centroids)
txt_centroids_np = np.array(txt_centroids)

# 최적의 클러스터 페어링 실행
matched_pairs = match_image_text_clusters(img_centroids_np, txt_centroids_np)

# 결과 출력
print(f"이미지-텍스트 클러스터 페어링:")
for img_idx, txt_idx in matched_pairs:
    print(f"   - 이미지 클러스터 {img_idx} ↔ 텍스트 클러스터 {txt_idx}")


# transformer 기반 cross-attention 모델

In [ ]:
class CrossAttentionTransformer(nn.Module):
    def __init__(self, input_dim, num_heads=4, num_layers=2):
        super().__init__()
        self.encoder_layer = nn.TransformerEncoderLayer(d_model=input_dim, nhead=num_heads)
        self.transformer_encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(input_dim, input_dim)  # 최종 Output Z 생성

    def forward(self, query, key, value):
        """
        Cross-Attention을 수행하여 Output Z를 생성
        Args:
            query (torch.Tensor): 클러스터 중심 벡터 (num_clusters, feature_dim)
            key (torch.Tensor): 클러스터 내 텍스트 벡터 (num_samples, feature_dim)
            value (torch.Tensor): 클러스터 내 텍스트 벡터 (num_samples, feature_dim)
        Returns:
            output_Z (torch.Tensor): Transformer를 통과한 최종 글로벌 표현 벡터
        """
        transformer_input = torch.cat([query, key], dim=0)  # Query와 Key 결합
        output = self.transformer_encoder(transformer_input)  # Transformer Encoder 통과
        output_Z = self.fc_out(output[:query.shape[0]])  # Query 크기에 맞는 Output Z 반환
        return output_Z


# 클러스터 개수만큼 Transformer 모델을 생성하고 Cross-Attention 수행

In [ ]:
# 동적으로 Transformer 모델 생성
transformers = [CrossAttentionTransformer(input_dim=512) for _ in range(matched_cluster_count)]

# Cross-Attention을 수행하여 Output Z 생성
output_Zs = []

for i in range(matched_cluster_count):
    transformer = transformers[i]

    # Query: 이미지 클러스터의 Centroid
    cluster_centroid = torch.tensor(img_centroids[i], dtype=torch.float32).unsqueeze(0)

    # Key & Value: 해당 클러스터의 텍스트 벡터들
    cluster_representations = torch.tensor(txt_vectors[i], dtype=torch.float32)

    # Transformer를 활용한 Cross-Attention 수행
    output_Z = transformer(cluster_centroid, cluster_representations, cluster_representations)
    output_Zs.append(output_Z)

print(f"총 {len(output_Zs)}개의 Output Z 생성 완료")


## Missing Modality를 Proxy Representation으로 보완하는 과정
- **클라이언트에 특정 모달리티(이미지/텍스트)가 없을 경우**

해당 클라이언트가 속한 이미지(텍스트) 클러스터에서 가장 가까운 다른 클라이언트를 찾음

이 다른 클라이언트가 속한 텍스트(이미지) 클러스터의 Centroid를 Proxy로 가져옴

Proxy Representation을 활용하여 Classification 진행

Z를 통해 로컬 모델을 업데이트하면, r_i, r_t도 자연스럽게 개선(예상)

- 문제:

연산 비용이 증가

클러스터의 Centroid를 Proxy로 사용하는 것이 항상 최적의 결과를 보장하지 않음

VS

## Missing modality 그대로 local model 만 Z 를 이용해 업데이트한 뒤 classify

연산 비용이 상대적으로 적음

로컬 모델을 업데이트하는 동시에 missing modality 에 대한 정보 부족이 어느 정도 보충됨

- 문제:

충분한 정도의 fusion 불가능

text/img 정보 없이 classify 하는 것이기 때문에 성능이 오히려 떨어지거나 큰 변화가 없을 수 있음

In [ ]:
import torch
from scipy.spatial.distance import cosine

def find_closest_proxy(client_vector, cluster_centroids):
    """
    가장 가까운 클러스터 중심 벡터(Proxy Representation)를 찾는 함수
    Args:
        client_vector (torch.Tensor): Missing Modality를 가진 클라이언트의 Representation
        cluster_centroids (torch.Tensor): 사용할 모달리티의 클러스터 중심 벡터 (num_clusters, feature_dim)
    Returns:
        proxy_vector (torch.Tensor): 가장 가까운 클러스터의 중심 벡터
    """
    min_distance = float("inf")
    best_centroid = None

    for centroid in cluster_centroids:
        distance = cosine(client_vector.cpu().numpy(), centroid.cpu().numpy())
        if distance < min_distance:
            min_distance = distance
            best_centroid = centroid

    return best_centroid

def update_local_model(local_model, output_Z, missing_modality, proxy_vector=None, alpha=0.5):
    """
    로컬 모델을 Output Z로 업데이트하고, Missing Modality가 있는 경우 Proxy Representation 활용
    Args:
        local_model (torch.nn.Module): 로컬 분류 모델
        output_Z (torch.Tensor): 글로벌 Transformer에서 생성된 Output Z
        missing_modality (bool): 특정 모달리티가 없는 경우 True
        proxy_vector (torch.Tensor or None): Proxy Representation 벡터 (선택적)
        alpha (float): Proxy Representation을 사용할 비율 (0~1)
    Returns:
        None
    """
    optimizer = torch.optim.Adam(local_model.parameters(), lr=0.001)

    # 기존 Representation 벡터 업데이트
    representation, logits = local_model(output_Z)

    # Missing Modality가 있는 경우 Proxy Representation 활용
    if missing_modality and proxy_vector is not None:
        representation = alpha * representation + (1 - alpha) * proxy_vector  # 가중치 조합

    # Classification Loss + Knowledge Distillation Loss
    loss = torch.nn.MSELoss()(representation, output_Z)  # Feature Alignment
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


# 연산 비용 줄이기

1. **모든 Missing Modality 클라이언트에 대해 연산하지 않고, 성능이 낮은 클라이언트만 Proxy Representation 사용**

  즉, 기존 Classification 성능이 너무 낮은 클라이언트만 보완

2. **Closest Representation을 찾을 때, 모든 클라이언트를 비교하지 않고, Top-K 후보군만 선택**

  예: 클러스터 내에서 가장 유사한 K개 클라이언트 중 랜덤 샘플링

## 사용하는 것이 유리한 경우(예상):

Missing Modality 클라이언트 비율이 높을 때

Representation이 충분히 구조화된 경우 (즉, 클러스터가 잘 형성된 경우)

클라이언트 간 Representation 유사성이 높은 경우

## 사용하는 것이 비효율적인 경우(예상):

Missing Modality가 일부 클라이언트에만 있는 경우 → 학습 성능 영향이 적음

Representation이 클러스터링되었지만, 클러스터 간 의미적 차이가 큰 경우

연산량이 너무 크거나, Proxy Representation이 도움이 되지 않는 경우

# Proxy Representation을 활용한 Local Model 업데이트 & Classification Pipeline

**Missing Modality를 Proxy Representation으로 보완하여 Local Model을 업데이트한 후, 업데이트된 Local Model로 다시 Classification을 수행**

- Step 1: 클라이언트가 Missing Modality 여부 확인
- Step 2: Missing Modality가 있는 경우, Proxy Representation을 찾음
- Step 3: Output 𝑍 와 Proxy Representation을 활용하여 Local Model 업데이트
- Step 4: 업데이트된 Local Model을 사용하여 다시 Classification 수행
- Step 5: 최종 Classification 결과 출력

# Proxy Representation 찾기

In [ ]:
import torch
from scipy.spatial.distance import cosine

def find_closest_proxy(client_vector, cluster_centroids):
    """
    가장 가까운 클러스터 중심 벡터(Proxy Representation)를 찾는 함수
    Args:
        client_vector (torch.Tensor): Missing Modality를 가진 클라이언트의 Representation
        cluster_centroids (torch.Tensor): 사용할 모달리티의 클러스터 중심 벡터 (num_clusters, feature_dim)
    Returns:
        proxy_vector (torch.Tensor): 가장 가까운 클러스터의 중심 벡터
    """
    min_distance = float("inf")
    best_centroid = None

    for centroid in cluster_centroids:
        distance = cosine(client_vector.cpu().numpy(), centroid.cpu().numpy())
        if distance < min_distance:
            min_distance = distance
            best_centroid = centroid

    return best_centroid


# local model 업데이트

In [ ]:
def update_local_model(local_model, output_Z, missing_modality, proxy_vector=None, alpha=0.5):
    """
    로컬 모델을 Output Z로 업데이트하고, Missing Modality가 있는 경우 Proxy Representation 활용
    Args:
        local_model (torch.nn.Module): 로컬 분류 모델
        output_Z (torch.Tensor): 글로벌 Transformer에서 생성된 Output Z
        missing_modality (bool): 특정 모달리티가 없는 경우 True
        proxy_vector (torch.Tensor or None): Proxy Representation 벡터 (선택적)
        alpha (float): Proxy Representation을 사용할 비율 (0~1)
    Returns:
        None
    """
    optimizer = torch.optim.Adam(local_model.parameters(), lr=0.001)

    # 기존 Representation 벡터 업데이트
    representation, logits = local_model(output_Z)

    # Missing Modality가 있는 경우 Proxy Representation 활용
    if missing_modality and proxy_vector is not None:
        representation = alpha * representation + (1 - alpha) * proxy_vector  # 가중치 조합

    # Classification Loss + Knowledge Distillation Loss
    loss = torch.nn.MSELoss()(representation, output_Z)  # Feature Alignment
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


In [ ]:
# 예제 클라이언트 설정
local_client = MultiModalClassifier(input_dim=512, hidden_dim=256, num_classes=10)
output_Z = torch.rand(1, 512)  # Transformer에서 나온 글로벌 Output Z
missing_modality = True  # 해당 클라이언트가 특정 모달리티(예: 텍스트)를 가지고 있지 않음

# Proxy Representation 찾기 (가장 가까운 텍스트 클러스터)
proxy_vector = find_closest_proxy(output_Z, txt_centroids_np) if missing_modality else None

# Local Model 업데이트
update_local_model(local_client, output_Z, missing_modality, proxy_vector)

# 업데이트된 Local Model로 Classification 수행
updated_representation, updated_logits = local_client(output_Z)

# Classification 결과 출력
predicted_class = torch.argmax(updated_logits, dim=-1)
print(f"클래스: {predicted_class.item()}")


# FC Layer를 활용하여 Multimodal Fusion을 수행

Z가 Image & Text Representation을 보완하도록 설계

𝑍 를 그대로 사용하는 대신, FC Layer를 통과시켜 Representation을 업데이트

# Gating Mechanism

- 각 Feature의 중요도를 조정하는 역할을 함.

- Z와 기존 Representation 중 어떤 정보를 더 반영할지 가중치를 학습하도록 도와줌.

- CNN의 SE (Squeeze-and-Excitation) Network나 Transformer의 Adaptive Fusion Mechanism에서도 활용됨.

https://medium.com/@eugenesh4work/gating-mechanisms-in-neural-networks-dc83a0bdb8c3  참고

In [ ]:
import torch.nn.functional as F

class FusionUpdate(nn.Module):
    """
    Local Model이 Z를 활용하여 Representation을 업데이트할 수 있도록 FC Layer를 활용한 Lightweight Fusion 적용.
    """
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.fc = nn.Linear(input_dim, hidden_dim)  # Lightweight Fusion Layer
        self.gate = nn.Linear(input_dim, hidden_dim)  # Gating Mechanism 추가

    def forward(self, Z, representation):
        """
        Args:
            Z (torch.Tensor): Global Model에서 생성된 Multimodal Fusion Output Z
            representation (torch.Tensor): Local Model에서 기존에 추출한 Representation

        Returns:
            updated_representation (torch.Tensor): FC Layer를 통해 업데이트된 Representation
        """
        fusion_weight = torch.sigmoid(self.gate(Z))  # 가중치 조정
        updated_representation = fusion_weight * self.fc(Z) + (1 - fusion_weight) * representation  # Adaptive Fusion
        return updated_representation


모델 자체에 fc layer 을 두고, 내부적인 메커니즘으로 gating 메커니즘을 써 업데이트를 한다.

초반에는 t-1, 그러니까 원래 Forget Layer 에 들어가야 할 Input z 자리니까 0 으로 시작

t, 즉 Input Layer 인 곳에는 이미지, 텍스트 rep이 late fusion 된 값이 들어간다.


Multiple works [14], [15] employ different fusion mechanisms such as averaging, voting schemes, variance etc. - 중요한 정보(모달리티에 따라 퓨전 메커니즘은 바뀔 수 있음)

그걸 t시점 input으로 해서 fc layer 에 통과시키고 classify task 를 수행한다.

그리고 Classification 과정에서 FC Layer의 가중치가 Backpropagation을 통해 업데이트됨.

# Local Model 업데이트 코드 (Lightweight Fusion 방식)

**Fully Connected Layer를 활용한 Lightweight Fusion**

Z를 분할하지 않고, 하나의 FC Layer를 통해 Image & Text Representation을 업데이트

FC Layer를 통해 𝑍 를 Local Model이 이해할 수 있는 Feature Space로 변환

Gating Weight를 활용하여 기존 Representation과 융합

### **Global Model의 Output 𝑍 는 Multimodal Fusion된 정보라서, 바로 Local Model이 이해하기 어려움**

- Local Model이 학습한 Feature Space(특징 공간)와 다를 수 있음

- 즉, Local Model이 원래 학습한 데이터 분포와 𝑍 가 맞지 않으면, 제대로 반영되지 않음

-> FC Layer를 사용하여 𝑍 를 Local Model이 이해할 수 있는 Feature Space로 변환

## FC Layer의 역할

1. **Feature Space 변환**

  𝑍 는 Global Model에서 나온 벡터지만, Local Model의 Feature Space와 다를 수 있음

  FC Layer를 사용해서 Local Model의 Feature Space와 맞추는 변환을 수행.

  즉, 𝑍 를 Local Representation과 같은 차원으로 변환함

2. **학습 가능한 변환 적용 가능**

  단순히 𝑍 를 사용하는 것이 아니라, 학습 가능한 가중치를 통해 Local Model에 최적화된 형태로 변환 가능

  모델이 점진적으로 𝑍 의 의미를 더 잘 이해하도록 학습됨

3. **Local Model의 기존 Representation과 융합 가능**

  𝑍 를 Local Model의 기존 Representation과 효과적으로 결합 가능

  단순히 합치는 것이 아니라, 가중치를 통해 최적의 비율을 학습할 수 있음

In [ ]:
def update_local_model(local_model, output_Z, missing_img=False, missing_txt=False,
                        proxy_img_vector=None, proxy_txt_vector=None, alpha=0.5):
    """
    Global Output Z를 Local Model에 FC Layer를 통해 주입하여 업데이트 (Cross-Attention 없이).

    Args:
        local_model (torch.nn.Module): 로컬 분류 모델
        output_Z (torch.Tensor): 글로벌 Transformer에서 생성된 Output Z (Multimodal Fusion)
        missing_img (bool): 이미지 모달리티가 없는 경우 True
        missing_txt (bool): 텍스트 모달리티가 없는 경우 True
        proxy_img_vector (torch.Tensor or None): 이미지 Proxy Representation 벡터
        proxy_txt_vector (torch.Tensor or None): 텍스트 Proxy Representation 벡터
        alpha (float): Proxy Representation을 사용할 비율 (0~1)

    Returns:
        None
    """
    optimizer = torch.optim.Adam(local_model.parameters(), lr=0.001)

    # Local Model에서 기존 Representation 추출
    img_rep, txt_rep, _, _ = local_model(None, None)

    # Missing Modality가 있는 경우 Proxy Representation 활용
    if missing_img and proxy_img_vector is not None:
        img_rep = alpha * img_rep + (1 - alpha) * proxy_img_vector  # 가중치 조합
    if missing_txt and proxy_txt_vector is not None:
        txt_rep = alpha * txt_rep + (1 - alpha) * proxy_txt_vector  # 가중치 조합

    # FC Layer 기반 Lightweight Fusion 수행
    fusion_module = FusionUpdate(input_dim=512, hidden_dim=256)

    if img_rep is not None:
        img_rep = fusion_module(output_Z, img_rep)
    if txt_rep is not None:
        txt_rep = fusion_module(output_Z, txt_rep)

    # Feature Alignment Loss (Multimodal Fusion을 유지)
    loss_img = torch.nn.MSELoss()(img_rep, output_Z) if img_rep is not None else 0
    loss_txt = torch.nn.MSELoss()(txt_rep, output_Z) if txt_rep is not None else 0
    total_loss = loss_img + loss_txt

    # Local Model 업데이트
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()


# VS 각 모달리티 별로 변환해서 업데이트 시키는 경우

In [ ]:
class FusionUpdate(nn.Module):
    """
    Local Model이 같은 Z를 사용하되, 이미지와 텍스트 Representation을 각각 다르게 업데이트할 수 있도록 설계.
    """
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.fc_img = nn.Linear(input_dim, hidden_dim)  # 이미지 전용 변환 레이어
        self.fc_txt = nn.Linear(input_dim, hidden_dim)  # 텍스트 전용 변환 레이어
        self.gate_img = nn.Linear(input_dim, hidden_dim)  # 이미지 Gating Mechanism
        self.gate_txt = nn.Linear(input_dim, hidden_dim)  # 텍스트 Gating Mechanism

    def forward(self, Z, img_representation=None, txt_representation=None):
        """
        Args:
            Z (torch.Tensor): Global Model에서 생성된 Multimodal Fusion Output Z
            img_representation (torch.Tensor): Local Model의 기존 이미지 Representation
            txt_representation (torch.Tensor): Local Model의 기존 텍스트 Representation

        Returns:
            updated_img_representation (torch.Tensor): 업데이트된 이미지 Representation
            updated_txt_representation (torch.Tensor): 업데이트된 텍스트 Representation
        """
        updated_img_rep, updated_txt_rep = None, None

        # 이미지 Representation 업데이트 (Z를 이미지에 맞게 변환)
        if img_representation is not None:
            img_weight = torch.sigmoid(self.gate_img(Z))  # Gating Mechanism을 통해 이미지 관련 정보만 선택
            transformed_Z_img = self.fc_img(Z)  # 이미지 전용 변환
            updated_img_rep = img_weight * transformed_Z_img + (1 - img_weight) * img_representation  # Adaptive Fusion

        # 텍스트 Representation 업데이트 (Z를 텍스트에 맞게 변환)
        if txt_representation is not None:
            txt_weight = torch.sigmoid(self.gate_txt(Z))  # Gating Mechanism을 통해 텍스트 관련 정보만 선택
            transformed_Z_txt = self.fc_txt(Z)  # 텍스트 전용 변환
            updated_txt_rep = txt_weight * transformed_Z_txt + (1 - txt_weight) * txt_representation  # Adaptive Fusion

        return updated_img_rep, updated_txt_rep